## Implementing the perceptron learning algorithm

This example shows how to implement the perceptron learning algorithm using NumPy. We implement the methods `fit` and `predict` so that our classifier can be used in the same way as any scikit-learn classifier. (See the [scikit-learn documentation](http://scikit-learn.org/stable/tutorial/statistical_inference/supervised_learning.html).)

The code uses a little bit of object-oriented programming. It's not anything particularly complicated, but if you're not used to object-oriented programming in Python, you might take a look at [this tutorial](https://python.swaroopch.com/oop.html).

In [1]:
import numpy as np

We first create a class that represents linear classifiers in general. This class does not have a `fit` method, because that will be implemented by subclasses representing specific learning algorithms for linear classifiers, e.g. the perceptron. So the thing we need to do here is to implement the `predict` method, because prediction works identically for all linear classifiers, regardless of how they were trained.

We also include a helper method `find_classes`, which finds the two output classes and associates them with positive and negative classifier scores, respectively.

In [2]:
class LinearClassifier(object):
    
    def find_classes(self, Y):
        """
        Finds the set of output classes in the output part Y of the training set.
        If there are exactly two classes, one of them is associated to positive
        classifier scores, the other one to negative scores. If the number of classes
        is not 2, an error is raised.
        """
        classes = sorted(set(Y))
        if len(classes) != 2:
            raise Exception("this does not seem to be a 2-class problem")
        self.positive_class = classes[0]
        self.negative_class = classes[1]
    
    def predict(self, X):        
        """
        Predicts the outputs for the inputs X. The inputs are assumed to be stored in
        a matrix, where each row contains the features for one instance.
        """

        # First compute the output scores
        scores = X.dot(self.w)

        # Select the positive or negative class label, depending on whether
        # the score was positive or negative.
        out = np.select([scores>=0.0, scores<=0.0], 
                        [self.positive_class, 
                         self.negative_class])
        return out

We now write the class that implements the perceptron learning algorithm. The actual learning algorithm is in the method called `fit`.

Note that this class has the same name as the `Perceptron` class in scikit-learn, so be careful when you import so that you don't get a name clash.

In [3]:
class Perceptron(LinearClassifier):
    
    def __init__(self, n_iter=10):
        """
        The constructor can optionally take a parameter n_iter specifying how 
        many times we want to iterate through the training set.
        """
        self.n_iter = n_iter

    def fit(self, X, Y):
        """
        Train a linear classifier using the perceptron learning algorithm.
        """
        
        # First determine which output class will be associated with positive
        # and negative scores, respectively.
        self.find_classes(Y)

        # Initialize the weight vector to all zeros.
        n_features = X.shape[1]
        self.w = np.zeros( n_features )

        for i in range(self.n_iter):            
            for x, y in zip(X, Y):
                
                # Compute the output score for this instance.
                score = x.dot(self.w)    

                # If there was an error, update the weights.
                if y == self.positive_class and score <= 0:
                    self.w += x
                if y == self.negative_class and score >= 0:
                    self.w -= x

### Testing our perceptron implementation on the Adult dataset

We will now test our perceptron implementation on the Adult dataset we have used in other contexts.

In [4]:
import pandas as pd

train_data = pd.read_csv('data/adult_train.csv')

n_cols = len(train_data.columns)
Xtrain = train_data.iloc[:, :n_cols-1].to_dict('records')
Ytrain = train_data.iloc[:, n_cols-1]

test_data = pd.read_csv('data/adult_test.csv')
Xtest = test_data.iloc[:, :n_cols-1].to_dict('records')
Ytest = test_data.iloc[:, n_cols-1]

To exemplify the instances in this dataset, let's print the input and output for the first instance. The input consists of a feature dictionary, containing named attributes such as `age`, `education` etc. The output is a string: in this case, either `'<=50K'` (low earner) or `'>50K'` (high earner). This is a *binary* classification problem because we have two output classes.

In [5]:
from pprint import pprint

pprint(Xtrain[0])
print()
print(Ytrain[0])

{'age': 27,
 'capital-gain': 0,
 'capital-loss': 0,
 'education': 'Some-college',
 'education-num': 10,
 'hours-per-week': 44,
 'marital-status': 'Divorced',
 'native-country': 'United-States',
 'occupation': 'Adm-clerical',
 'race': 'White',
 'relationship': 'Unmarried',
 'sex': 'Female',
 'workclass': 'Private'}

<=50K


Now let's assemble the building blocks.

In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

# The DictVectorizer is used to map symbolic features to numerical vectors.
# Note that we set sparse=False, because our Perceptron implementation assumes
# that the examples are stored in a NumPy matrix.
dv = DictVectorizer(sparse=False)

# A StandardScaler divides the features by their standard deviation. The purpuse is that
# the numerical features should have a similar magnitude.
sc = StandardScaler(with_mean=False)

# Make an instance of the perceptron class we implemented above.
clf = Perceptron(n_iter=5)

# Combine the vectorizer, scaler and the classifier into a pipeline.
pipeline = make_pipeline( dv, sc, clf )

# Train the classifier, evaluate on the test set.
pipeline.fit(Xtrain, Ytrain);

And finally run the classifier on the test set and compute its accuracy.

In [7]:
Yguess = pipeline.predict(Xtest)
accuracy_score(Ytest, Yguess)

0.8275290215588723

### Inspecting the features learned by the classifier

Now, let's take a look at what the perceptron algorithm has come up with. First, let's see which category corresponds to the positive scores, and which to the negative scores.

In [8]:
clf.positive_class, clf.negative_class

('<=50K', '>50K')

This means that positive scores will be interpreted as the category `<=50K`, and negative scores as `>50K`.

We will then see which features the learning algorithm has assigned high weights to. We can first just look at the weights stored in the weight vector `w`, that we built in the `fit` method that we created previously. This is a long vector, so we'll just print the first 10 dimensions. We see that the first three features have negative weights, which shows that an increase in these features will increase our certainty that this person is a high earner (negative classifier score). The other seven features point in the other direction: increasing them makes the classifier think that this person is a low earner.

In [9]:
clf.w[:10]

array([ -10.70363725, -102.33862078,   -6.6980345 ,    6.60789315,
         17.98217841,   16.08540761,    8.7299537 ,   13.95781725,
         39.75748753,    7.17107147])

The result above didn't tell us that much, really, because it's not obvious how to interpret the positions. To understand the meaning of each position, we need to look into the `DictVectorizer` that we used to map named features into a feature matrix.

In a `DictVectorizer`, this information is stored in the attribute called `feature_names_`. The feature names appear in the same order as they do in the weight vector. So this means that the first column in the feature matrix is `age`. The second feature, `capital-gain`, has a much stronger association with the negative class.

In [10]:
dv.feature_names_[:10]

['age',
 'capital-gain',
 'capital-loss',
 'education-num',
 'education=10th',
 'education=11th',
 'education=12th',
 'education=1st-4th',
 'education=5th-6th',
 'education=7th-8th']

We print the 20 features that have the highest negative weights. In this case, the negative class is `>50K`, or the people who earned more than $50,000 a year. As you can see, features look quite meaningful: for instance, people who own capital or have a college degree are more likely to have a high income.

(If you wonder about the functions `sorted` and `zip`, please take a look at the [documentation of Python built-in functions](https://docs.python.org/3/library/functions.html).)

In [11]:
for weight, fname in sorted( zip(clf.w, dv.feature_names_) )[:20]:
    print(fname, weight)

capital-gain -102.33862077830157
marital-status=Married-civ-spouse -22.07096476462833
native-country=Italy -21.143399510818224
occupation=Exec-managerial -18.150181273381424
education=Masters -17.867837119203056
hours-per-week -17.16983033211865
native-country=Germany -15.449134542485275
education=Prof-school -15.17201665308389
native-country=Philippines -12.862955066670706
occupation=Sales -12.678846283296279
age -10.703637250799328
education=Doctorate -8.93605284999826
capital-loss -6.698034499014547
occupation=Tech-support -6.009710553364639
relationship=Wife -4.6708174816320565
education=Bachelors -2.6976502940177483
workclass=Self-emp-inc -1.7763568394002505e-15
relationship=Unmarried -8.881784197001252e-16
education=Assoc-acdm 0.0
marital-status=Married-AF-spouse 0.0


Conversely, the features most strongly associated with the positive class (`<=50K`, low earners) also tend to be meaningful, such as being unemployed or not having an education.

In [12]:
for weight, fname in sorted( zip(clf.w, dv.feature_names_), reverse=True)[:20]:
    print(fname, weight)

education=Preschool 75.86219705663098
occupation=Armed-Forces 60.15720376953795
workclass=Without-pay 48.2367717040825
native-country=Outlying-US(Guam-USVI-etc) 48.23677170407935
native-country=Thailand 42.54344835057305
education=5th-6th 39.75748752843618
native-country=Ecuador 34.11588658978079
marital-status=Never-married 34.07746604654098
native-country=Peru 32.424617062898236
native-country=Nicaragua 30.96251791883376
native-country=Portugal 29.682118601074002
native-country=Iran 27.536046929149357
native-country=Haiti 27.221757482047522
native-country=Columbia 23.51346786853978
sex=Female 23.379423147363756
native-country=Poland 23.317057692391852
native-country=Japan 22.938599065898213
native-country=Vietnam 22.067785039426557
relationship=Own-child 22.067765083545023
marital-status=Divorced 20.392262097390656
